# AeroPulse PM2.5 — Phase 7: Tail-Aware ML Experiments

Find the training configuration that stops the model underpredicting hazardous air, and
measure what each new feature group actually bought.

## Objective

Phase 5 established the defect precisely. On the `extreme` regime (PM2.5 ≥ 150 µg/m³) the
model's mean signed error was **−103 µg/m³**, and extreme recall at a 24-hour horizon was
**zero**. The model was not merely inaccurate on hazardous air; it systematically
predicted clean air during episodes.

That is a loss-function problem before it is a feature problem. Roughly 1% of rows are
extreme, so an unweighted squared-error fit spends almost all of its gradient on air
nobody needs a warning about, and the optimum is to hedge toward the mean.

### The headline metric is extreme bias, not RMSE

This is the most important convention in the notebook. **RMSE and extreme bias disagree
by construction**: a model can improve RMSE by hedging harder toward the mean, which
makes extreme bias worse. Ranking on RMSE is how Phase 5 arrived at a model that looked
reasonable and could not raise an alarm.

Every table below therefore leads with `extreme_bias`, and every row is reported against
the **persistence baseline** as `AGENTS.md` requires.

### What is being tested

| | Experiment |
|---|---|
| E0 | baseline, unweighted squared error |
| E1 | `log1p` target |
| E2 | tail weighting by concentration band |
| E3 | `log1p` + tail weighting |
| E4 | Huber objective |
| E5 | quantile regression at q10 / q50 / q90 |
| E6 | v3 physical features vs the v2-equivalent subset |
| E7 | direct per-horizon models, 1 h to 48 h |
| E8 | 24-hour peak target |
| E9 | hazard classifier |
| E10 | probability calibration |

| | Ablation (cumulative) |
|---|---|
| A0 | temporal only |
| A1 | + weather |
| A2 | + fire |
| A3 | + regional |
| A4 | + satellite |
| A5 | + static |
| A6 | + calendar |

### Validation

Purged rolling-origin folds from `phase7_eval.py`, which is the same split rule Phase 5
used — so these numbers are comparable to the baselines they are meant to beat. Random
splits are forbidden (LLD §19): the features are autoregressive and hourly, so a shuffled
test row's answer sits in the training set as a feature of its neighbour.

In [1]:
# ============================================================================
# SETUP
# ============================================================================
from pathlib import Path
import os, json, time, warnings, gc
import numpy as np
import pandas as pd
from dotenv import find_dotenv, load_dotenv

from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

import phase7_eval as ev

_dotenv = find_dotenv(usecwd=True)
if _dotenv:
    load_dotenv(_dotenv, override=True)
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
EVENT_DIR = PROJECT_ROOT / "data" / "pm25" / "processed" / "event_aware"
EVENT_FILE = EVENT_DIR / "pm25_event_aware_features.parquet"
MANIFEST_FILE = EVENT_DIR / "dataset_manifest.json"
OUT_DIR = PROJECT_ROOT / "artifacts" / "pm25" / "phase7" / "experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The operational horizon. The roadmap's acceptance gate asks for >= 24h median
# lead time, so 24h is where the loss-function comparison has to hold up. E7
# sweeps the rest.
PRIMARY_H = int(os.getenv("PM25_PRIMARY_H", "24"))
# 5 folds, not 3, and the reason is not statistical power. rolling_origin_folds
# tests on the LAST test_frac * n_splits of the span, so 3 folds at 0.12 test
# only the final 36% - which on this dataset begins in mid-February and contains
# no winter and no post-monsoon. Those are the crop-burning and inversion seasons
# where the extreme episodes live, so 3 folds would measure an extreme-regime fix
# on the one part of the year that has almost no extremes. 5 folds reaches back
# to October and matches the Phase-5 evaluation, keeping the two comparable.
N_SPLITS = int(os.getenv("PM25_CV_SPLITS", "5"))
EMBARGO_H = 48

# One architecture for every experiment, so a difference between rows is
# attributable to the loss or the features and not to capacity. 400 trees is
# where the holdout score stops improving on this dataset.
LGBM_BASE = dict(n_estimators=400, learning_rate=0.05, num_leaves=63,
                 min_child_samples=40, subsample=0.9, subsample_freq=1,
                 colsample_bytree=0.8, random_state=42, verbose=-1, n_jobs=-1)

if not EVENT_FILE.exists():
    raise FileNotFoundError(f"{EVENT_FILE} not found. Run Phase 2 first.")
manifest = json.loads(MANIFEST_FILE.read_text())
if manifest["dataset_version"] != "3.0.0":
    raise ValueError(f"expected dataset 3.0.0, found {manifest['dataset_version']}")

print(f"dataset      : v{manifest['dataset_version']}  "
      f"{manifest['rows']:,} rows, {manifest['n_features']} features")
print(f"fingerprint  : {manifest['sha256_16']}")
print(f"primary horizon: t+{PRIMARY_H}h    folds: {N_SPLITS}")

dataset      : v3.0.0  1,705,252 rows, 184 features
fingerprint  : ba72d91a76c075ee
primary horizon: t+24h    folds: 5


## 1. Load into a dense matrix

The feature list comes from the Phase-2 manifest, never from a hand-written list — that
is the rule in `AGENTS.md`, and it is what keeps this notebook honest if Phase 2 changes.

**No imputer.** Phase 5 wrapped every model in `SimpleImputer(strategy="median")`. That
is actively wrong for this feature set: `fire_min_distance_5km_24h` is NaN for 95% of
rows because *there is no fire within 5 km*, and median-imputing it asserts a fire at a
typical distance. LightGBM routes NaN down its own branch at each split, which preserves
"absent" as the distinct signal it is.

In [2]:
# ============================================================================
# LOAD - dense float32 matrix, integer-indexed folds
# ============================================================================
# Held as one matrix rather than a DataFrame because each fold otherwise copies
# ~1 GB of pandas blocks; integer indexing into a contiguous array is both
# cheaper and what LightGBM wants anyway.
# ============================================================================
FEATURES = manifest["features"]
GROUPS = manifest["feature_groups"]
TARGETS = manifest["targets"]
META_COLS = ["timestamp_utc", "location_id", "season", "pollution_regime"]

t0 = time.time()
df = pd.read_parquet(EVENT_FILE, columns=FEATURES + TARGETS + META_COLS)
X = df[FEATURES].to_numpy("float32")
Y = {t: df[t].to_numpy("float32") for t in TARGETS}
meta = df[META_COLS].reset_index(drop=True)
TS = pd.DatetimeIndex(df["timestamp_utc"])
CUR = df["pm25"].to_numpy("float64")          # persistence baseline input
FEAT_POS = {f: i for i, f in enumerate(FEATURES)}
del df
gc.collect()

print(f"Loaded in {time.time()-t0:.0f}s")
print(f"X            : {X.shape}  {X.nbytes/1e9:.2f} GB float32")
print(f"NaN cells    : {np.isnan(X).mean():.1%} (kept as NaN - see above)")
print(f"Span         : {TS.min()} -> {TS.max()}")
print("\nFeature groups available to the ablation:")
for g, cols in GROUPS.items():
    print(f"  {g:12} {len(cols):>3}")

Loaded in 1s
X            : (1705252, 184)  1.26 GB float32


NaN cells    : 6.0% (kept as NaN - see above)
Span         : 2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00

Feature groups available to the ablation:
  temporal      42
  weather       28
  fire          46
  regional      15
  satellite     22
  calendar      18
  static         7
  interaction    6


### Folds

Five checks are printed per fold. The one that matters is the extreme count in the test
block: extreme recall computed on a handful of rows is noise, and the burning season
falls in only one of these folds because the dataset covers a single post-monsoon cycle.
That is a real limitation of the data, and it is stated per fold rather than averaged away.

In [3]:
# ============================================================================
# ROLLING-ORIGIN FOLDS - purged, expanding window
# ============================================================================
TARGET_MAIN = f"target_pm25_t_plus_{PRIMARY_H}h"
valid_main = ~np.isnan(Y[TARGET_MAIN])
idx_main = np.flatnonzero(valid_main)

FOLDS = list(ev.rolling_origin_folds(TS[idx_main], n_splits=N_SPLITS,
                                     purge_hours=PRIMARY_H, embargo_hours=EMBARGO_H))
# Map fold positions (relative to idx_main) back onto the full matrix.
FOLDS = [(idx_main[tr], idx_main[te], cut, end) for tr, te, cut, end in FOLDS]

print(f"{len(FOLDS)} folds on target {TARGET_MAIN} "
      f"(purge {PRIMARY_H}h + embargo {EMBARGO_H}h)\n")
rows = []
for i, (tr, te, cut, end) in enumerate(FOLDS, 1):
    y_te = Y[TARGET_MAIN][te]
    rows.append({
        "fold": i, "train_rows": len(tr), "test_rows": len(te),
        "test_start": str(pd.Timestamp(cut).date()),
        "test_end": str(pd.Timestamp(end).date()),
        "test_extreme_rows": int((y_te >= 150).sum()),
        "test_extreme_rate": float((y_te >= 150).mean()),
        "test_mean_pm25": float(np.mean(y_te)),
        "dominant_season": meta.loc[te, "season"].mode().iloc[0],
    })
fold_table = pd.DataFrame(rows)
display(fold_table)

thin = fold_table[fold_table.test_extreme_rows < 200]
if len(thin):
    print(f"CAUTION: folds {list(thin.fold)} have fewer than 200 extreme test rows. "
          "Their extreme_recall and extreme_bias are indicative only.")

# ---------------------------------------------------------------------------
# SEASONAL COVERAGE GUARD
# ---------------------------------------------------------------------------
# This notebook exists to fix an extreme-regime bias, and the extreme regime is
# a post-monsoon and winter phenomenon. If no test block lands in those seasons
# then every extreme metric below is measured where extremes barely occur, and
# the whole comparison is answering the wrong question while still producing
# confident-looking numbers. That is a silent failure, so it is made loud.
# ---------------------------------------------------------------------------
HIGH_SEASONS = {"post_monsoon", "winter"}
covered = set(fold_table.dominant_season) & HIGH_SEASONS
total_extreme = int(fold_table.test_extreme_rows.sum())
print(f"\nTest-block seasons : {list(fold_table.dominant_season)}")
print(f"High-pollution seasons covered: {sorted(covered) or 'NONE'}")
print(f"Extreme rows across all test blocks: {total_extreme:,}")
if not covered:
    raise RuntimeError(
        "No test fold lands in post-monsoon or winter, so extreme_bias and "
        "extreme_recall would be measured almost entirely on clean air. Raise "
        f"PM25_CV_SPLITS (currently {N_SPLITS}) so the folds reach back into the "
        "burning season, or shorten test_frac.")
print("\nThe dataset spans one post-monsoon cycle, so the burning season appears in "
      "only part of this sequence. Fold-to-fold spread below is therefore partly "
      "seasonal composition, not just model instability.")

5 folds on target target_pm25_t_plus_24h (purge 24h + embargo 48h)



,fold,train_rows,test_rows,test_start,test_end,test_extreme_rows,test_extreme_rate,test_mean_pm25,dominant_season
0,1,625627,194597,2025-10-03,2025-12-10,6018,0.030925,56.122475,post_monsoon
1,2,819354,160736,2025-12-10,2026-02-16,5770,0.035897,62.575211,winter
2,3,983771,187438,2026-02-16,2026-04-25,1316,0.007021,39.206959,pre_monsoon
3,4,1168025,176624,2026-04-25,2026-07-02,610,0.003454,28.535454,pre_monsoon
4,5,1345168,180709,2026-07-02,2026-09-08,382,0.002114,21.378414,monsoon



Test-block seasons : ['post_monsoon', 'winter', 'pre_monsoon', 'pre_monsoon', 'monsoon']
High-pollution seasons covered: ['post_monsoon', 'winter']
Extreme rows across all test blocks: 14,096

The dataset spans one post-monsoon cycle, so the burning season appears in only part of this sequence. Fold-to-fold spread below is therefore partly seasonal composition, not just model instability.


## 2. The experiment harness

One function runs every experiment, so the differences between rows come from the
configuration under test rather than from accidental differences in how each was
evaluated. Persistence is scored on the identical test rows every time.

In [4]:
# ============================================================================
# EXPERIMENT HARNESS
# ============================================================================
def run_experiment(label, target, cols=None, weight_fn=None, log_target=False,
                   model_kwargs=None, folds=None, notes=""):
    """Fit and score one configuration across every fold.

    Args:
        label: experiment id used in the results table.
        target: target column name.
        cols: feature subset; None means every feature.
        weight_fn: y_train -> sample weights, for the tail-weighted runs.
        log_target: fit on log1p(y) and invert with expm1 for scoring. Metrics are
            always reported in ug/m3, never in log space, so they stay comparable.
        model_kwargs: overrides on LGBM_BASE (objective, alpha, ...).
        notes: free text carried into the artifact.

    Returns:
        (per_fold DataFrame, aggregate dict)
    """
    cols = cols or FEATURES
    col_idx = np.array([FEAT_POS[c] for c in cols])
    params = {**LGBM_BASE, **(model_kwargs or {})}
    yv = Y[target]
    per_fold, t_start = [], time.time()

    for i, (tr, te, cut, end) in enumerate(folds or FOLDS, 1):
        # Rows whose target is missing cannot train or score. Dropped per fold so
        # that a target with a shorter tail does not shrink the other folds.
        tr = tr[~np.isnan(yv[tr])]
        te = te[~np.isnan(yv[te])]
        if len(tr) < 500 or len(te) < 100:
            continue
        y_tr, y_te = yv[tr].astype("float64"), yv[te].astype("float64")
        fit_y = np.log1p(np.clip(y_tr, 0, None)) if log_target else y_tr
        w = weight_fn(y_tr) if weight_fn else None

        model = LGBMRegressor(**params)
        model.fit(X[np.ix_(tr, col_idx)], fit_y, sample_weight=w)
        raw = model.predict(X[np.ix_(te, col_idx)])
        pred = np.expm1(raw) if log_target else raw
        pred = np.clip(pred, 0, None)          # a negative concentration is not a forecast

        base = ev.persistence(CUR[te])
        m = ev.regression_metrics(y_te, pred)
        e = ev.event_metrics(y_te, pred)
        bm = ev.regression_metrics(y_te, base)
        be = ev.event_metrics(y_te, base)
        per_fold.append({
            "experiment": label, "fold": i, "test_rows": len(te),
            **m, **{k: v for k, v in e.items() if k != "threshold"},
            "skill_vs_persistence": ev.skill_score(m["RMSE"], bm["RMSE"]),
            "persistence_RMSE": bm["RMSE"],
            "persistence_extreme_recall": be["extreme_recall"],
            "persistence_extreme_bias": be["extreme_bias"],
        })
        del model
        gc.collect()

    pf = pd.DataFrame(per_fold)
    if pf.empty:
        return pf, {"experiment": label, "note": "no usable fold"}
    num = pf.select_dtypes("number")
    agg = {"experiment": label, "target": target, "n_features": len(cols),
           "folds": len(pf), "notes": notes,
           # The loss configuration is recorded as DATA, exactly as it was run.
           # Notebooks 07 and 08 rebuild the winning model from these fields.
           # They must not infer it by parsing the experiment name: several
           # experiments share a configuration under different labels, and
           # "E7_h24" - which is a tail-weighted run - contains no substring that
           # would reveal that. A name-parsing consumer would silently train an
           # unweighted model while reporting the weighted one's metrics.
           "cfg_log_target": bool(log_target),
           "cfg_tail_weights": weight_fn is not None,
           # "" rather than None: a None here becomes NaN once these dicts are
           # stacked into a DataFrame, and NaN then serialises into the report as
           # a bare `nan` token that strict JSON parsers reject.
           "cfg_objective": params.get("objective") or "",
           "cfg_full_feature_set": len(cols) == len(FEATURES),
           "cfg_quantile": (float(params["alpha"])
                            if params.get("objective") == "quantile" else None)}
    for c in ["R2", "MAE", "RMSE", "bias", "extreme_bias", "extreme_recall",
              "extreme_precision", "extreme_f1", "skill_vs_persistence",
              "persistence_extreme_bias", "persistence_extreme_recall"]:
        agg[c] = float(num[c].mean())
    # Fold-to-fold spread is a first-class result: a configuration that wins on
    # average but swings wildly is not deployable.
    agg["R2_std"] = float(num["R2"].std())
    agg["extreme_bias_std"] = float(num["extreme_bias"].std())
    agg["seconds"] = round(time.time() - t_start, 1)
    print(f"  {label:26} extreme_bias {agg['extreme_bias']:+8.1f}  "
          f"recall {agg['extreme_recall']:.3f}  R2 {agg['R2']:+.4f}  "
          f"RMSE {agg['RMSE']:6.2f}  ({agg['seconds']:.0f}s)")
    return pf, agg


ALL_FOLDS, ALL_AGG = [], []


def record(pf, agg):
    if not pf.empty:
        ALL_FOLDS.append(pf)
    ALL_AGG.append(agg)


print("Harness ready. Every run is scored against persistence on identical test rows.")

Harness ready. Every run is scored against persistence on identical test rows.


## 3. E0–E5: does the loss function fix the tail?

Six ways of telling the model that hazardous air matters more than clean air.

`log1p` (E1) is included because it is the reflex answer to a skewed target, and it is
worth measuring rather than assuming: compressing the target also compresses the
*penalty* for missing a peak, so it can make underprediction in µg/m³ worse even as it
improves R² in log space. Metrics here are always reported in µg/m³.

In [5]:
# ============================================================================
# E0-E4  loss-function experiments
# ============================================================================
print(f"Target {TARGET_MAIN}, {len(FOLDS)} folds. Headline column is extreme_bias:\n")

record(*run_experiment("E0_baseline", TARGET_MAIN,
                       notes="unweighted L2, the Phase-5 configuration"))
record(*run_experiment("E1_log1p", TARGET_MAIN, log_target=True,
                       notes="fit log1p(y), invert for scoring"))
record(*run_experiment("E2_tail_weights", TARGET_MAIN, weight_fn=ev.tail_weights,
                       notes="sample weights 1/1.5/2/4/8/12 by concentration band"))
record(*run_experiment("E3_log_plus_weights", TARGET_MAIN, log_target=True,
                       weight_fn=ev.tail_weights, notes="E1 and E2 combined"))
record(*run_experiment("E4_huber", TARGET_MAIN,
                       model_kwargs={"objective": "huber", "alpha": 5.0},
                       notes="Huber; robust to outliers, which may be the wrong "
                             "instinct when the outliers ARE the signal"))

Target target_pm25_t_plus_24h, 5 folds. Headline column is extreme_bias:



  E0_baseline                extreme_bias   -155.1  recall 0.117  R2 +0.3711  RMSE  25.21  (202s)


  E1_log1p                   extreme_bias   -171.6  recall 0.073  R2 +0.3206  RMSE  26.44  (224s)


  E2_tail_weights            extreme_bias   -135.3  recall 0.207  R2 +0.2346  RMSE  27.50  (198s)


  E3_log_plus_weights        extreme_bias   -154.7  recall 0.119  R2 +0.3609  RMSE  25.44  (167s)


  E4_huber                   extreme_bias   -178.4  recall 0.000  R2 +0.3183  RMSE  26.66  (213s)


In [6]:
# ============================================================================
# E5  quantile regression - q10 / q50 / q90
# ============================================================================
# Why a quantile model belongs in an alerting product: the q90 model answers
# "how bad could this plausibly get?", which is the question an operator is
# actually asking. A conditional-mean model answers "what is the expected
# value?", and the expected value of a skewed distribution sits below the
# outcome that needs the warning.
# ============================================================================
q_pred_store = {}
for q in (0.1, 0.5, 0.9):
    pf, agg = run_experiment(f"E5_quantile_q{int(q*100)}", TARGET_MAIN,
                             model_kwargs={"objective": "quantile", "alpha": q},
                             notes=f"asymmetric pinball loss at q={q}")
    record(pf, agg)

e5 = pd.DataFrame([a for a in ALL_AGG if a["experiment"].startswith("E5")])
if not e5.empty:
    print("\nThe q90 model is expected to overpredict on average (positive bias) and "
          "to recover extreme recall - that is the trade it exists to make, not a "
          "defect. Compare its extreme_bias against E0's:")
    print(e5[["experiment", "bias", "extreme_bias", "extreme_recall",
              "extreme_precision", "RMSE"]].to_string(index=False))

  E5_quantile_q10            extreme_bias   -202.1  recall 0.000  R2 -0.0571  RMSE  33.40  (163s)


  E5_quantile_q50            extreme_bias   -166.6  recall 0.074  R2 +0.3576  RMSE  25.50  (192s)


  E5_quantile_q90            extreme_bias   -115.6  recall 0.346  R2 +0.0410  RMSE  30.94  (166s)

The q90 model is expected to overpredict on average (positive bias) and to recover extreme recall - that is the trade it exists to make, not a defect. Compare its extreme_bias against E0's:
     experiment       bias  extreme_bias  extreme_recall  extreme_precision      RMSE
E5_quantile_q10 -18.620552   -202.074600        0.000000           0.000000 33.395987
E5_quantile_q50  -3.892161   -166.590704        0.074377           0.256758 25.498934
E5_quantile_q90  16.274233   -115.645695        0.346022           0.252629 30.942130


## 4. E6: what did the new physical features buy?

E0–E5 change the loss. E6 changes the information: the same configuration is fitted on
the v3 feature set and on a v2-equivalent subset, which is the direct measurement of
whether upwind fire attribution, CAMS composition, terrain and the festival calendar
were worth acquiring.

In [7]:
# ============================================================================
# E6  v3 features vs the v2-equivalent subset
# ============================================================================
# The v2 subset is reconstructed as "temporal + weather + calendar-basics only":
# v2 had no upwind attribution, no composition, no terrain and no festival
# table. This is an approximation of v2, not a re-run of it, and is labelled as
# such - v2's exact columns no longer exist on disk.
V2_LIKE = sorted(set(GROUPS["temporal"]) | set(GROUPS["weather"]) |
                 {c for c in GROUPS["calendar"]
                  if c in ("hour", "dow", "doy", "month", "is_weekend",
                           "sin_hour", "cos_hour", "sin_doy", "cos_doy")})
NEW_ONLY = sorted(set(FEATURES) - set(V2_LIKE))

print(f"v2-equivalent : {len(V2_LIKE)} features")
print(f"added in v3   : {len(NEW_ONLY)} features\n")
record(*run_experiment("E6a_v2_like_features", TARGET_MAIN, cols=V2_LIKE,
                       weight_fn=ev.tail_weights,
                       notes="approximates the v2 feature set; tail-weighted"))
record(*run_experiment("E6b_v3_all_features", TARGET_MAIN, cols=FEATURES,
                       weight_fn=ev.tail_weights,
                       notes="full v3 feature set; tail-weighted"))

a = next(r for r in ALL_AGG if r["experiment"] == "E6a_v2_like_features")
b = next(r for r in ALL_AGG if r["experiment"] == "E6b_v3_all_features")
print(f"\nAdding the v3 driver features moved:")
print(f"  extreme_bias   {a['extreme_bias']:+8.1f} -> {b['extreme_bias']:+8.1f} ug/m3")
print(f"  extreme_recall {a['extreme_recall']:8.3f} -> {b['extreme_recall']:8.3f}")
print(f"  R2             {a['R2']:+8.4f} -> {b['R2']:+8.4f}")
print(f"  RMSE           {a['RMSE']:8.2f} -> {b['RMSE']:8.2f} ug/m3")

v2-equivalent : 79 features
added in v3   : 105 features



  E6a_v2_like_features       extreme_bias   -134.9  recall 0.220  R2 +0.2371  RMSE  27.39  (82s)


  E6b_v3_all_features        extreme_bias   -135.3  recall 0.207  R2 +0.2346  RMSE  27.50  (134s)

Adding the v3 driver features moved:
  extreme_bias     -134.9 ->   -135.3 ug/m3
  extreme_recall    0.220 ->    0.207
  R2              +0.2371 ->  +0.2346
  RMSE              27.39 ->    27.50 ug/m3


## 5. E7: how far ahead is the forecast actually useful?

Skill decays with lead time. Where it crosses persistence is the product's usable
warning horizon, and it is the number a public-health user needs before trusting an
alert. Each horizon gets its own folds, because the purge width depends on the horizon.

In [8]:
# ============================================================================
# E7  direct per-horizon models
# ============================================================================
# One model per horizon rather than one model extrapolated across horizons: a
# +1h forecast is nearly persistence while a +48h forecast is a different
# problem, and a single model forced to serve both underfits each.
horizon_rows = []
for h in [1, 3, 6, 12, 24, 48]:
    target_h = f"target_pm25_t_plus_{h}h"
    if target_h not in Y:
        continue
    ok = np.flatnonzero(~np.isnan(Y[target_h]))
    folds_h = [(ok[tr], ok[te], cut, end) for tr, te, cut, end in
               ev.rolling_origin_folds(TS[ok], n_splits=N_SPLITS,
                                       purge_hours=h, embargo_hours=EMBARGO_H)]
    pf, agg = run_experiment(f"E7_h{h:02d}", target_h, weight_fn=ev.tail_weights,
                             folds=folds_h, notes=f"direct model at t+{h}h")
    record(pf, agg)
    horizon_rows.append({"horizon_h": h, **{k: agg[k] for k in
                        ["R2", "MAE", "RMSE", "extreme_bias", "extreme_recall",
                         "extreme_precision", "skill_vs_persistence",
                         "persistence_extreme_recall"]}})

horizons = pd.DataFrame(horizon_rows)
display(horizons)

useful = horizons[horizons.skill_vs_persistence > 0]
if len(useful):
    print(f"Beats persistence on RMSE out to {useful.horizon_h.max()}h "
          f"(best {useful.skill_vs_persistence.max():.1%} at "
          f"{useful.loc[useful.skill_vs_persistence.idxmax(),'horizon_h']:.0f}h).")
else:
    print("WARNING: never beats persistence on RMSE at any horizon.")
beats_recall = horizons[horizons.extreme_recall > horizons.persistence_extreme_recall]
print(f"Beats persistence on EXTREME RECALL at horizons: "
      f"{list(beats_recall.horizon_h) or 'none'}")
print("\nThese two rankings can disagree, and when they do the recall one is the "
      "operationally relevant answer: persistence has perfect recall at +1h and no "
      "skill at +24h, which is exactly the range an alerting product lives in.")

  E7_h01                     extreme_bias    -56.7  recall 0.660  R2 +0.6895  RMSE  16.21  (130s)


  E7_h03                     extreme_bias   -109.9  recall 0.328  R2 +0.4227  RMSE  23.00  (138s)


  E7_h06                     extreme_bias   -126.9  recall 0.250  R2 +0.3001  RMSE  26.21  (141s)


  E7_h12                     extreme_bias   -139.3  recall 0.191  R2 +0.2642  RMSE  26.82  (135s)


  E7_h24                     extreme_bias   -135.3  recall 0.207  R2 +0.2346  RMSE  27.50  (126s)


  E7_h48                     extreme_bias   -149.2  recall 0.129  R2 +0.1511  RMSE  29.46  (132s)


,horizon_h,R2,MAE,RMSE,extreme_bias,extreme_recall,extreme_precision,skill_vs_persistence,persistence_extreme_recall
0,1,0.689498,6.548768,16.205560,-56.689515,0.660385,0.622428,0.069702,0.632064
1,3,0.422660,12.055726,22.996383,-109.881250,0.328251,0.420259,0.164936,0.336762
2,6,0.300075,14.950936,26.206279,-126.934089,0.249560,0.332409,0.203904,0.222179
3,12,0.264245,15.440215,26.822561,-139.267644,0.191021,0.286474,0.225588,0.170086
4,24,0.234554,16.064990,27.498373,-135.336030,0.206677,0.296306,0.102022,0.290979
5,48,0.151104,17.601494,29.461526,-149.207847,0.128911,0.223257,0.115570,0.232693


Beats persistence on RMSE out to 48h (best 22.6% at 12h).
Beats persistence on EXTREME RECALL at horizons: [1, 6, 12]

These two rankings can disagree, and when they do the recall one is the operationally relevant answer: persistence has perfect recall at +1h and no skill at +24h, which is exactly the range an alerting product lives in.


## 6. E8–E10: peak and hazard

The concentration forecast answers "how dirty at 14:00 tomorrow". Neither an operator
nor a citizen asks that. They ask "will it be dangerous tomorrow", and Phase 5 showed
that thresholding a regression to answer it produced **zero** extreme recall at 24 hours.

These three experiments size the alternative. Notebook 07 builds it properly, with
calibration, thresholds and latency benchmarks; here the question is only whether the
approach is worth building.

In [9]:
# ============================================================================
# E8  peak target - the max over the next 24h rather than one future hour
# ============================================================================
# Why the peak is the better regression target for alerting: an episode that
# peaks at 03:00 matters whether or not 03:00 happens to be the hour being
# scored. Predicting a specific future hour makes the model's job harder AND
# less useful at the same time.
record(*run_experiment("E8_peak_next_24h", "target_max_pm25_next_24h",
                       weight_fn=ev.tail_weights,
                       notes="target is max PM2.5 over t+1..t+24"))

peak = next(r for r in ALL_AGG if r["experiment"] == "E8_peak_next_24h")
conc = next(r for r in ALL_AGG if r["experiment"] == "E6b_v3_all_features")
print(f"\nPeak target vs point-in-time target (same features, same loss):")
print(f"  extreme_recall  {conc['extreme_recall']:.3f} -> {peak['extreme_recall']:.3f}")
print(f"  extreme_bias    {conc['extreme_bias']:+.1f} -> {peak['extreme_bias']:+.1f}")
print("  (the two targets are different quantities, so R2 is not comparable "
      "across them - recall and bias are what transfer)")

  E8_peak_next_24h           extreme_bias   -102.4  recall 0.547  R2 +0.1627  RMSE  61.62  (135s)

Peak target vs point-in-time target (same features, same loss):
  extreme_recall  0.207 -> 0.547
  extreme_bias    -135.3 -> -102.4
  (the two targets are different quantities, so R2 is not comparable across them - recall and bias are what transfer)


In [10]:
# ============================================================================
# E9 / E10  hazard classifier and probability calibration
# ============================================================================
# A classifier optimises the decision directly instead of inferring it from a
# regression that was never trained for it. scale_pos_weight compensates for the
# 6% base rate.
#
# CALIBRATION MATTERS OPERATIONALLY: a model whose "0.8" means 0.3 in practice
# cannot be given a threshold an operator can reason about, however good its
# ranking is. Platt and isotonic are fitted on a held-out CALIBRATION SLICE cut
# from the END of the training block - never on the test block, which would be
# fitting on the evaluation data.
# ============================================================================
HAZ_TARGET = "target_extreme_within_24h"
haz_rows, cal_rows = [], []

for i, (tr, te, cut, end) in enumerate(FOLDS, 1):
    yv = Y[HAZ_TARGET]
    tr = tr[~np.isnan(yv[tr])]
    te = te[~np.isnan(yv[te])]
    if len(tr) < 5000 or len(te) < 500 or len(np.unique(yv[te])) < 2:
        continue
    # Calibration slice: the last 15% of the training block by time. Taking it
    # from the end keeps the fit chronological, and it is still strictly before
    # the purge gap, so nothing from the test period is used.
    ord_tr = tr[np.argsort(TS[tr].to_numpy())]
    n_cal = int(len(ord_tr) * 0.15)
    fit_idx, cal_idx = ord_tr[:-n_cal], ord_tr[-n_cal:]
    y_fit, y_cal, y_te = yv[fit_idx], yv[cal_idx], yv[te]
    if y_fit.sum() < 50 or y_cal.sum() < 20:
        continue

    pos_w = float((len(y_fit) - y_fit.sum()) / max(y_fit.sum(), 1))
    clf = LGBMClassifier(**{**LGBM_BASE, "scale_pos_weight": pos_w})
    clf.fit(X[fit_idx], y_fit)

    p_raw_cal, p_raw_te = clf.predict_proba(X[cal_idx])[:, 1], clf.predict_proba(X[te])[:, 1]
    platt = LogisticRegression(max_iter=1000).fit(p_raw_cal.reshape(-1, 1), y_cal)
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_raw_cal, y_cal)
    variants = {"raw": p_raw_te,
                "platt": platt.predict_proba(p_raw_te.reshape(-1, 1))[:, 1],
                "isotonic": iso.predict(p_raw_te)}

    # Threshold swept on the CALIBRATION slice only. Choosing it on the test
    # block would report a score no deployment could reproduce.
    for name, p_te in variants.items():
        p_cal = {"raw": p_raw_cal,
                 "platt": platt.predict_proba(p_raw_cal.reshape(-1, 1))[:, 1],
                 "isotonic": iso.predict(p_raw_cal)}[name]
        grid = np.linspace(0.05, 0.95, 19)
        f1s = [ev.classification_metrics(y_cal, p_cal, t)["f1"] for t in grid]
        thr = float(grid[int(np.argmax(f1s))])
        cal_rows.append({"variant": name, "fold": i, "chosen_threshold": thr,
                         **ev.classification_metrics(y_te, p_te, thr),
                         "calibration_error": ev.calibration_error(y_te, p_te)})

    base = ev.majority_class_baseline(y_te)
    haz_rows.append({"fold": i, "test_rows": len(te),
                     "base_rate": float(y_te.mean()),
                     "majority_recall": base["recall"], "majority_f1": base["f1"],
                     "majority_brier": base["brier"]})
    del clf
    gc.collect()

hazard = pd.DataFrame(cal_rows)
if hazard.empty:
    print("No fold had enough hazard positives to train a classifier.")
else:
    summary = (hazard.groupby("variant")[
        ["chosen_threshold", "precision", "recall", "f1", "pr_auc", "roc_auc",
         "brier", "calibration_error"]].mean().round(4))
    print("E9/E10 - hazard classifier, mean over folds:")
    display(summary)
    b = pd.DataFrame(haz_rows)
    print(f"Majority-class baseline: recall {b.majority_recall.mean():.3f}, "
          f"f1 {b.majority_f1.mean():.3f}, brier {b.majority_brier.mean():.4f}")
    print(f"Base rate in test blocks: {b.base_rate.mean():.2%}")
    best_cal = summary["calibration_error"].idxmin()
    print(f"\nBest calibrated variant: {best_cal} "
          f"(ECE {summary.loc[best_cal, 'calibration_error']:.4f})")
    print(f"Best PR-AUC            : {summary['pr_auc'].idxmax()} "
          f"({summary['pr_auc'].max():.4f}) vs base rate {b.base_rate.mean():.4f}")
    print("\nPR-AUC leads over ROC-AUC here because the positive class is rare: "
          "ROC-AUC looks flattering on an imbalanced problem, and PR-AUC against "
          "the base rate is the honest lift.")

E9/E10 - hazard classifier, mean over folds:


,chosen_threshold,precision,recall,f1,pr_auc,roc_auc,brier,calibration_error
variant,,,,,,,,
isotonic,0.23,0.4298,0.4744,0.4273,0.4363,0.8464,0.0558,0.0256
platt,0.23,0.4445,0.4627,0.4416,0.4535,0.8479,0.0558,0.0275
raw,0.52,0.4338,0.4722,0.4292,0.4535,0.8479,0.0702,0.0795


Majority-class baseline: recall 0.000, f1 0.000, brier 0.0728
Base rate in test blocks: 8.32%

Best calibrated variant: isotonic (ECE 0.0256)
Best PR-AUC            : platt (0.4535) vs base rate 0.0832

PR-AUC leads over ROC-AUC here because the positive class is rare: ROC-AUC looks flattering on an imbalanced problem, and PR-AUC against the base rate is the honest lift.


## 7. A0–A6: cumulative feature ablation

Groups are added in order of acquisition cost — temporal features are free, satellite
and calendar data had to be built. The question each row answers is whether that cost
bought anything on the metric that matters.

Because the groups are cumulative and disjoint (Phase 2 asserts the partition), a
group's contribution is the change from the previous row.

In [11]:
# ============================================================================
# A0-A6  cumulative feature-group ablation
# ============================================================================
ABLATION_ORDER = ["temporal", "weather", "fire", "regional", "satellite",
                  "static", "calendar"]
missing = [g for g in ABLATION_ORDER if g not in GROUPS]
if missing:
    raise ValueError(f"manifest is missing feature groups {missing}")

print("Cumulative ablation, tail-weighted, same folds:\n")
cum, abl_rows = [], []
for k, g in enumerate(ABLATION_ORDER):
    cum += GROUPS[g]
    label = f"A{k}_{'+'.join(ABLATION_ORDER[:k+1]) if k == 0 else 'plus_' + g}"
    pf, agg = run_experiment(label, TARGET_MAIN, cols=sorted(set(cum)),
                             weight_fn=ev.tail_weights,
                             notes=f"cumulative through '{g}'")
    record(pf, agg)
    abl_rows.append({"step": f"A{k}", "added_group": g, "n_features": agg["n_features"],
                     **{c: agg[c] for c in ["R2", "MAE", "RMSE", "extreme_bias",
                                            "extreme_recall", "extreme_precision"]}})

ablation = pd.DataFrame(abl_rows)
for c in ["R2", "RMSE", "extreme_bias", "extreme_recall"]:
    ablation[f"d_{c}"] = ablation[c].diff()
display(ablation)

print("Marginal contribution of each group to extreme_bias (positive = less "
      "underprediction, which is the improvement we want):")
for r in ablation.itertuples():
    if not np.isnan(r.d_extreme_bias):
        print(f"  + {r.added_group:10} {r.d_extreme_bias:+8.2f} ug/m3   "
              f"(recall {r.d_extreme_recall:+.4f}, R2 {r.d_R2:+.4f})")
worst = ablation.dropna(subset=["d_extreme_bias"]).nsmallest(1, "d_extreme_bias")
if len(worst) and worst.iloc[0].d_extreme_bias < 0:
    print(f"\nNote: adding '{worst.iloc[0].added_group}' made extreme bias worse. "
          "More features is not monotonically better - a group that only helps the "
          "bulk of the distribution can push the fit further from the tail.")

Cumulative ablation, tail-weighted, same folds:



  A0_temporal                extreme_bias   -134.9  recall 0.224  R2 +0.2343  RMSE  27.39  (48s)


  A1_plus_weather            extreme_bias   -133.8  recall 0.237  R2 +0.2451  RMSE  27.19  (71s)


  A2_plus_fire               extreme_bias   -135.6  recall 0.216  R2 +0.2534  RMSE  27.04  (93s)


  A3_plus_regional           extreme_bias   -134.5  recall 0.221  R2 +0.2498  RMSE  27.26  (104s)


  A4_plus_satellite          extreme_bias   -134.4  recall 0.225  R2 +0.2460  RMSE  27.34  (114s)


  A5_plus_static             extreme_bias   -135.2  recall 0.215  R2 +0.2472  RMSE  27.28  (118s)


  A6_plus_calendar           extreme_bias   -136.1  recall 0.207  R2 +0.2326  RMSE  27.37  (129s)


,step,added_group,n_features,R2,MAE,RMSE,extreme_bias,extreme_recall,extreme_precision,d_R2,d_RMSE,d_extreme_bias,d_extreme_recall
0,A0,temporal,42,0.234311,15.798280,27.388942,-134.898718,0.224081,0.299472,NaN,NaN,NaN,NaN
1,A1,weather,70,0.245148,15.783357,27.187954,-133.808298,0.236910,0.318016,0.010838,-0.200988,1.090420,0.012829
2,A2,fire,116,0.253445,15.560030,27.041616,-135.591257,0.215573,0.313386,0.008297,-0.146338,-1.782959,-0.021338
3,A3,regional,131,0.249793,15.699092,27.256073,-134.536615,0.221157,0.297364,-0.003652,0.214457,1.054642,0.005585
4,A4,satellite,153,0.245996,15.771329,27.335091,-134.417901,0.224663,0.303693,-0.003797,0.079018,0.118714,0.003505
5,A5,static,160,0.247158,15.741213,27.276259,-135.159157,0.215469,0.295232,0.001161,-0.058832,-0.741256,-0.009193
6,A6,calendar,178,0.232555,15.993222,27.372812,-136.083987,0.207006,0.302901,-0.014603,0.096553,-0.924830,-0.008464


Marginal contribution of each group to extreme_bias (positive = less underprediction, which is the improvement we want):
  + weather       +1.09 ug/m3   (recall +0.0128, R2 +0.0108)
  + fire          -1.78 ug/m3   (recall -0.0213, R2 +0.0083)
  + regional      +1.05 ug/m3   (recall +0.0056, R2 -0.0037)
  + satellite     +0.12 ug/m3   (recall +0.0035, R2 -0.0038)
  + static        -0.74 ug/m3   (recall -0.0092, R2 +0.0012)
  + calendar      -0.92 ug/m3   (recall -0.0085, R2 -0.0146)

Note: adding 'fire' made extreme bias worse. More features is not monotonically better - a group that only helps the bulk of the distribution can push the fit further from the tail.


## 8. Results, and what carries forward

The ranking rule is stated explicitly rather than left to the reader: **smallest
|extreme_bias| first, then highest extreme recall, then RMSE**. Ranking on RMSE is what
produced the Phase-5 model that could not raise an alarm.

**Ranking is not selection.** The table below ranks everything that was run, but two
classes of row must not be promoted from it.

*Configurations that lose to persistence.* `AGENTS.md` requires every model to beat an
honest baseline, and a configuration can win on extreme bias while being worse overall —
predicting a high number everywhere reduces tail underprediction and ruins everything else.
That is a real outcome here, not a hypothetical: the q90 quantile model has the best
extreme bias in the table and *negative* skill against persistence.

*Feature-subset diagnostics.* The A0–A6 ablations and E6a are fitted on a subset to measure
what each feature group contributes. Notebooks 07 and 08 train on the full set, so promoting
an ablation would attach a 70-feature run's metrics to a 184-feature model.

Selection therefore filters to full-feature configurations that beat persistence, then ranks
those by extreme bias. Everything excluded is named, with its numbers, rather than dropped
quietly.

In [12]:
# ============================================================================
# CONSOLIDATED RESULTS
# ============================================================================
results = pd.DataFrame(ALL_AGG)
results["abs_extreme_bias"] = results["extreme_bias"].abs()
# mergesort is stable. Several experiments are the same configuration under
# different labels (E2, E6b and E7_h24 are all tail-weighted runs on the primary
# target), so they tie exactly on every sort key. With an unstable sort the
# winner's NAME would vary between runs even though the chosen model does not,
# which would make the artifact non-reproducible for no reason.
ranked = results.sort_values(["abs_extreme_bias", "extreme_recall", "RMSE"],
                             ascending=[True, False, True], kind="mergesort")

view = ["experiment", "target", "n_features", "extreme_bias", "extreme_bias_std",
        "extreme_recall", "extreme_precision", "R2", "R2_std", "RMSE",
        "skill_vs_persistence", "seconds"]
print("Ranked by |extreme_bias|, then extreme recall, then RMSE:")
display(ranked[view].round(4).reset_index(drop=True))

# Compare like with like: only the runs on the primary target and horizon.
same_target = ranked[ranked.target == TARGET_MAIN]

# ---------------------------------------------------------------------------
# BASELINE ELIGIBILITY - AGENTS.md, non-negotiable
# ---------------------------------------------------------------------------
# "A model that cannot beat its baseline must not be promoted." Skill is
# 1 - RMSE_model/RMSE_persistence, so a positive value means the model is closer
# to the truth than repeating the last observation. Anything at or below zero is
# not a forecast, whatever its extreme bias looks like.
# ---------------------------------------------------------------------------
# Ablations and the reduced-feature runs are DIAGNOSTICS, not candidates. They
# exist to measure what each feature group contributes, and they are fitted on a
# subset. Notebooks 07 and 08 train on the full feature set, so selecting an
# ablation would attach a 70-feature run's metrics to a 184-feature model and
# report them as that model's. Only full-feature runs are eligible.
candidates = same_target[same_target.cfg_full_feature_set]
diagnostics = same_target[~same_target.cfg_full_feature_set]
if len(diagnostics):
    print(f"Not candidates ({len(diagnostics)} feature-subset diagnostics: "
          f"{', '.join(sorted(diagnostics.experiment))}). They are fitted on a "
          "subset, so their metrics do not describe the model that ships.\n")

eligible = candidates[candidates.skill_vs_persistence > 0]
rejected = candidates[candidates.skill_vs_persistence <= 0]
if len(rejected):
    print("EXCLUDED - ranked well but failed the persistence baseline:")
    for r in rejected.itertuples():
        print(f"  {r.experiment:22} extreme_bias {r.extreme_bias:+8.1f}  "
              f"skill {r.skill_vs_persistence:+.3f}  R2 {r.R2:+.3f}")
    print("  These reduce tail underprediction by predicting high everywhere, "
          "which costs more than it buys. AGENTS.md forbids promoting them.\n")
if eligible.empty:
    raise RuntimeError(
        "No configuration on the primary target beats persistence. Nothing may "
        "be promoted; the hazard classifier in notebook 07 is the route that "
        "does not depend on winning this comparison.")

best = eligible.iloc[0]
e0 = results[results.experiment == "E0_baseline"].iloc[0]
print(f"Selected: {best.experiment} - best extreme bias among the "
      f"{len(eligible)} full-feature configurations that beat persistence.\n")
print(f"\nPhase-5-equivalent baseline (E0): extreme_bias {e0.extreme_bias:+.1f}, "
      f"recall {e0.extreme_recall:.3f}, R2 {e0.R2:+.4f}")
print(f"Best on {TARGET_MAIN}          : {best.experiment} -> "
      f"extreme_bias {best.extreme_bias:+.1f}, recall {best.extreme_recall:.3f}, "
      f"R2 {best.R2:+.4f}")
print(f"Extreme underprediction reduced by "
      f"{abs(e0.extreme_bias) - abs(best.extreme_bias):+.1f} ug/m3 "
      f"({(abs(e0.extreme_bias)-abs(best.extreme_bias))/abs(e0.extreme_bias):+.1%})")
print(f"Persistence on the same rows     : extreme_bias "
      f"{best.persistence_extreme_bias:+.1f}, recall "
      f"{best.persistence_extreme_recall:.3f}")
if best.extreme_recall <= best.persistence_extreme_recall:
    print("\nPERSISTENCE STILL MATCHES OR BEATS THE BEST MODEL ON EXTREME RECALL. "
          "Per AGENTS.md a model that cannot beat its baseline must not be "
          "promoted; the hazard classifier in notebook 07 is the route that does "
          "not depend on winning this particular comparison.")

Ranked by |extreme_bias|, then extreme recall, then RMSE:


,experiment,target,n_features,extreme_bias,extreme_bias_std,extreme_recall,extreme_precision,R2,R2_std,RMSE,skill_vs_persistence,seconds
0,E7_h01,target_pm25_t_plus_1h,184,-56.6895,44.0125,0.6604,0.6224,0.6895,0.1712,16.2056,0.0697,129.9
1,E8_peak_next_24h,target_max_pm25_next_24h,184,-102.4355,61.2545,0.5473,0.3832,0.1627,0.1512,61.6182,0.0931,135.0
2,E7_h03,target_pm25_t_plus_3h,184,-109.8812,67.2939,0.3283,0.4203,0.4227,0.2221,22.9964,0.1649,137.8
3,E5_quantile_q90,target_pm25_t_plus_24h,184,-115.6457,87.8271,0.3460,0.2526,0.0410,0.2082,30.9421,-0.0112,165.6
4,E7_h06,target_pm25_t_plus_6h,184,-126.9341,65.7238,0.2496,0.3324,0.3001,0.1814,26.2063,0.2039,141.1
5,A1_plus_weather,target_pm25_t_plus_24h,70,-133.8083,76.4109,0.2369,0.3180,0.2451,0.2024,27.1880,0.1100,71.3
6,A4_plus_satellite,target_pm25_t_plus_24h,153,-134.4179,76.2168,0.2247,0.3037,0.2460,0.1805,27.3351,0.1070,114.5
7,A3_plus_regional,target_pm25_t_plus_24h,131,-134.5366,76.7853,0.2212,0.2974,0.2498,0.1802,27.2561,0.1093,104.2
8,A0_temporal,target_pm25_t_plus_24h,42,-134.8987,74.4817,0.2241,0.2995,0.2343,0.2032,27.3889,0.1032,48.4
9,E6a_v2_like_features,target_pm25_t_plus_24h,79,-134.9086,75.0473,0.2200,0.3179,0.2371,0.2128,27.3936,0.1057,81.5


Not candidates (8 feature-subset diagnostics: A0_temporal, A1_plus_weather, A2_plus_fire, A3_plus_regional, A4_plus_satellite, A5_plus_static, A6_plus_calendar, E6a_v2_like_features). They are fitted on a subset, so their metrics do not describe the model that ships.

EXCLUDED - ranked well but failed the persistence baseline:
  E5_quantile_q90        extreme_bias   -115.6  skill -0.011  R2 +0.041
  E5_quantile_q10        extreme_bias   -202.1  skill -0.076  R2 -0.057
  These reduce tail underprediction by predicting high everywhere, which costs more than it buys. AGENTS.md forbids promoting them.

Selected: E2_tail_weights - best extreme bias among the 8 full-feature configurations that beat persistence.


Phase-5-equivalent baseline (E0): extreme_bias -155.1, recall 0.117, R2 +0.3711
Best on target_pm25_t_plus_24h          : E2_tail_weights -> extreme_bias -135.3, recall 0.207, R2 +0.2346
Extreme underprediction reduced by +19.8 ug/m3 (+12.7%)
Persistence on the same rows     : extre

In [13]:
# ============================================================================
# SEASONAL AND SPATIAL BREAKDOWN of the winning configuration
# ============================================================================
# An aggregate can hide that the model works in the monsoon and fails in the
# burning season - which is the season the product exists for.
# ============================================================================
best_label = best.experiment
best_cfg = {
    "E0_baseline": dict(),
    "E1_log1p": dict(log_target=True),
    "E2_tail_weights": dict(weight_fn=ev.tail_weights),
    "E3_log_plus_weights": dict(log_target=True, weight_fn=ev.tail_weights),
    "E4_huber": dict(model_kwargs={"objective": "huber", "alpha": 5.0}),
    "E5_quantile_q10": dict(model_kwargs={"objective": "quantile", "alpha": 0.1}),
    "E5_quantile_q50": dict(model_kwargs={"objective": "quantile", "alpha": 0.5}),
    "E5_quantile_q90": dict(model_kwargs={"objective": "quantile", "alpha": 0.9}),
    "E6b_v3_all_features": dict(weight_fn=ev.tail_weights),
}.get(best_label, dict(weight_fn=ev.tail_weights))

# Refit on the last fold only, purely to obtain predictions for the breakdown.
tr, te, _, _ = FOLDS[-1]
yv = Y[TARGET_MAIN]
tr, te = tr[~np.isnan(yv[tr])], te[~np.isnan(yv[te])]
params = {**LGBM_BASE, **best_cfg.get("model_kwargs", {})}
y_tr = yv[tr].astype("float64")
fit_y = np.log1p(np.clip(y_tr, 0, None)) if best_cfg.get("log_target") else y_tr
w = best_cfg["weight_fn"](y_tr) if best_cfg.get("weight_fn") else None
m = LGBMRegressor(**params).fit(X[tr], fit_y, sample_weight=w)
raw = m.predict(X[te])
pred = np.clip(np.expm1(raw) if best_cfg.get("log_target") else raw, 0, None)
y_te = yv[te].astype("float64")

print(f"Breakdown for {best_label} on the final fold "
      f"({pd.Timestamp(TS[te].min()).date()} -> {pd.Timestamp(TS[te].max()).date()}):\n")
print("By pollution regime (this is the table Phase 5's -103 ug/m3 came from):")
display(ev.regime_metrics(y_te, pred).round(3))
print("By season:")
season_bd = ev.breakdown(meta.loc[te].reset_index(drop=True), y_te, pred, "season")
display(season_bd.round(3))

# Spatial view: per-station errors, summarised. A single mean hides a handful of
# stations the model cannot fit at all.
st = ev.breakdown(meta.loc[te].reset_index(drop=True), y_te, pred, "location_id",
                  min_rows=100)
if not st.empty:
    print(f"Per-station MAE across {len(st)} stations: "
          f"median {st.MAE.median():.2f}, p90 {st.MAE.quantile(0.9):.2f}, "
          f"worst {st.MAE.max():.2f} ug/m3")
    print(f"Stations with negative R2 (worse than predicting their own mean): "
          f"{int((st.R2 < 0).sum())}/{len(st)}")
persist_regime = ev.regime_metrics(y_te, ev.persistence(CUR[te]))
print("\nPersistence on the same rows, by regime:")
display(persist_regime.round(3))
del m
gc.collect()

Breakdown for E2_tail_weights on the final fold (2026-07-02 -> 2026-09-08):

By pollution regime (this is the table Phase 5's -103 ug/m3 came from):


,regime,count,MAE,RMSE,bias
0,normal,148283,8.474,13.344,6.594
1,moderate,29084,10.279,15.293,-0.677
2,high,2150,24.698,29.836,-15.119
3,very_high,816,44.740,52.981,-34.744
4,extreme,376,208.352,261.551,-208.323


By season:


,season,rows,MAE,RMSE,R2,bias,extreme_recall,extreme_bias
0,monsoon,180709,9.537,18.691,0.065,4.531,0.045,-205.982


Per-station MAE across 149 stations: median 7.63, p90 17.41, worst 57.95 ug/m3
Stations with negative R2 (worse than predicting their own mean): 126/149

Persistence on the same rows, by regime:


,regime,count,MAE,RMSE,bias
0,normal,148283,6.330,14.912,1.739
1,moderate,29084,11.953,19.872,-5.198
2,high,2150,32.679,44.556,-20.195
3,very_high,816,56.183,67.904,-38.334
4,extreme,376,215.785,271.309,-194.243


0

In [14]:
# ============================================================================
# PERSIST PHASE-7 EXPERIMENT ARTIFACTS
# ============================================================================
per_fold_all = pd.concat(ALL_FOLDS, ignore_index=True) if ALL_FOLDS else pd.DataFrame()
per_fold_all.to_csv(OUT_DIR / "experiment_folds.csv", index=False)
ranked[view].to_csv(OUT_DIR / "experiment_summary.csv", index=False)
ablation.to_csv(OUT_DIR / "ablation_cumulative.csv", index=False)
horizons.to_csv(OUT_DIR / "horizon_sweep.csv", index=False)
fold_table.to_csv(OUT_DIR / "fold_definition.csv", index=False)
if not hazard.empty:
    hazard.to_csv(OUT_DIR / "hazard_calibration_folds.csv", index=False)
season_bd.to_csv(OUT_DIR / "best_config_by_season.csv", index=False)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "dataset": {"version": manifest["dataset_version"],
                "fingerprint": manifest["sha256_16"],
                "rows": manifest["rows"], "n_features": manifest["n_features"]},
    "protocol": {
        "validation": "purged rolling-origin CV (expanding window)",
        "n_splits": len(FOLDS),
        "purge_hours": PRIMARY_H,
        "embargo_hours": EMBARGO_H,
        "random_split": "FORBIDDEN - autoregressive features leak across a shuffle "
                        "(LLD section 19)",
        "primary_horizon_h": PRIMARY_H,
        "ranking_rule": "|extreme_bias| asc, then extreme_recall desc, then RMSE asc",
        "selection_rule": "rank as above, but only among configurations with "
                          "skill_vs_persistence > 0; ranking orders the table, "
                          "the baseline filter decides what may be promoted",
        "baseline": "persistence for regression, majority class for classification",
        "imputation": "none - LightGBM branches on NaN, which preserves 'no fire "
                      "nearby' as a distinct signal that median imputation destroys",
    },
    "model_architecture": LGBM_BASE,
    "experiments": ALL_AGG,
    "ablation": ablation.to_dict("records"),
    "horizon_sweep": horizons.to_dict("records"),
    "hazard_calibration": (hazard.groupby("variant").mean(numeric_only=True)
                           .reset_index().to_dict("records")
                           if not hazard.empty else None),
    "best_on_primary_target": {
        "experiment": best.experiment,
        "extreme_bias": float(best.extreme_bias),
        "extreme_recall": float(best.extreme_recall),
        "R2": float(best.R2), "RMSE": float(best.RMSE),
        "skill_vs_persistence": float(best.skill_vs_persistence),
        "beats_persistence_on_extreme_recall":
            bool(best.extreme_recall > best.persistence_extreme_recall),
        # Consumers rebuild the model from THIS, not from the label above.
        "loss_config": {
            "log_target": bool(best.cfg_log_target),
            "tail_weights": bool(best.cfg_tail_weights),
            "objective": best.cfg_objective or None,
            "quantile": (float(best.cfg_quantile)
                         if pd.notna(best.cfg_quantile) else None),
        },
        "selection_pool": "full-feature runs on the primary target that beat "
                          "persistence; ablations are diagnostics, not candidates",
        "selection_rule": "smallest |extreme_bias| AMONG configurations with "
                          "skill_vs_persistence > 0 (AGENTS.md: a model that "
                          "cannot beat its baseline must not be promoted)",
        "excluded_for_failing_baseline": [
            {"experiment": r.experiment,
             "extreme_bias": float(r.extreme_bias),
             "skill_vs_persistence": float(r.skill_vs_persistence)}
            for r in rejected.itertuples()],
    },
    "baseline_E0": {"extreme_bias": float(e0.extreme_bias),
                    "extreme_recall": float(e0.extreme_recall),
                    "R2": float(e0.R2), "RMSE": float(e0.RMSE)},
    "caveats": [
        "The dataset spans one winter and one post-monsoon cycle, so seasonal "
        "generalisation is asserted on a single instance of each. Fold-to-fold "
        "spread mixes model instability with seasonal composition.",
        "Weather features are observations at t, not forecasts valid at t+h. A "
        "deployed long-horizon forecast would use forecast weather, so these "
        "long-horizon numbers are conservative rather than optimistic.",
        "E6a approximates the v2 feature set from v3 groups; it is not a re-run "
        "of the v2 pipeline.",
    ],
}
with open(OUT_DIR / "experiment_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print("PHASE-7 EXPERIMENTS COMPLETE\n")
print(f"  runs        : {len(ALL_AGG)}")
print(f"  best config : {best.experiment}")
print(f"  extreme bias: {e0.extreme_bias:+.1f} (E0) -> {best.extreme_bias:+.1f} ug/m3")
print(f"\nWrote to {OUT_DIR}:")
for p in sorted(OUT_DIR.glob("*")):
    print(f"  {p.name:38} {p.stat().st_size/1e3:8.1f} KB")
print("\nNotebook 07 builds the peak and hazard models; notebook 08 tunes and "
      "checks the acceptance gates.")

PHASE-7 EXPERIMENTS COMPLETE

  runs        : 24
  best config : E2_tail_weights
  extreme bias: -155.1 (E0) -> -135.3 ug/m3

Wrote to /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase7/experiments:
  ablation_cumulative.csv                     1.5 KB
  best_config_by_season.csv                   0.2 KB
  experiment_folds.csv                       30.7 KB
  experiment_report.json                     32.1 KB
  experiment_summary.csv                      4.9 KB
  fold_definition.csv                         0.6 KB
  hazard_calibration_folds.csv                3.0 KB
  horizon_sweep.csv                           1.1 KB

Notebook 07 builds the peak and hazard models; notebook 08 tunes and checks the acceptance gates.
